In [51]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_validate
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score




from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor



In [52]:
df = pd.read_csv("../Data/processed/smartphones_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())

Dataset shape: (1020, 31)
                      model  \
0              OPPO F35 Pro   
1  Samsung Galaxy S25 Ultra   
2             OPPO K14 Plus   
3   Motorola Edge 70 Fusion   
4               Vivo V80 5G   

                                                 url     brand scrape_date  \
0                             https://smpx.to/jifOik      OPPO  2026-10-14   
1  https://www.smartprix.com/mobiles/samsung-gala...   Samsung  2026-10-14   
2  https://www.smartprix.com/mobiles/oppo-k14-plu...      OPPO  2026-10-14   
3  https://www.smartprix.com/mobiles/motorola-edg...  Motorola  2026-10-14   
4  https://www.smartprix.com/mobiles/vivo-v80-5g-...      Vivo  2026-10-14   

   price_inr  spec_score                 processor_name  cpu_cores  cpu_ghz  \
0      45999          80             Dimensity 7360 Max        8.0     2.50   
1      94999          92  Snapdragon 8 Elite for Galaxy        8.0     4.47   
2      25999          75             Dimensity 7360 Max        8.0     2.50   
3 

In [53]:
# Select only the records intended for the primary analysis
ml_df = df.loc[df["in_analysis_scope"] == True].copy()

# Define the target
target = "price_inr"

print("ML dataset shape:", ml_df.shape)
print("Missing target values:", ml_df[target].isna().sum())
print("Target price summary:")
print(ml_df[target].describe())

ML dataset shape: (838, 31)
Missing target values: 0
Target price summary:
count       838.000000
mean      44113.464200
std       46450.424966
min        6499.000000
25%       18999.000000
50%       29999.000000
75%       49990.000000
max      449900.000000
Name: price_inr, dtype: float64


In [54]:
# Target: smartphone listed price in INR
ml_df["price_inr"] = pd.to_numeric(ml_df["price_inr"], errors="coerce")
print("ML rows:", len(ml_df))
print("Missing prices:", ml_df["price_inr"].isna().sum())
print(ml_df["price_inr"].describe())

assert ml_df["price_inr"].notna().all()

ML rows: 838
Missing prices: 0
count       838.000000
mean      44113.464200
std       46450.424966
min        6499.000000
25%       18999.000000
50%       29999.000000
75%       49990.000000
max      449900.000000
Name: price_inr, dtype: float64


In [55]:
# Numeric predictors
numeric_features = [
    "spec_score",
    "ram_gb",
    "storage_gb",
    "cpu_cores",
    "cpu_ghz",
    "battery_mah",
    "fast_charge_w",
    "screen_in",
    "res_short",
    "res_long",
    "refresh_hz",
    "rear_main_mp",
    "rear_mp_value_count",
    "front_mp",
]

# Categorical predictors
categorical_features = [
    "brand",
    "processor_name",
    "os_family",
    "charging_status",
    "is_foldable",
    "is_5g",
    "has_nfc",
    "has_esim",
    "has_memory_card"
]




In [56]:
# Select features
features = numeric_features + categorical_features

X = ml_df[features].copy()
y = ml_df["price_inr"].copy()

print("X shape:", X.shape)
print("Features:", len(features))

X shape: (838, 23)
Features: 23


In [57]:
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(features))

Numeric features: 14
Categorical features: 9
Total features: 23


In [58]:

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 670
Testing rows: 168


In [59]:
#  Preprocess numeric and categorical features separately
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("string_converter", FunctionTransformer(
        lambda X: X.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features) ,
    
] ,  sparse_threshold=0)

In [60]:

# 4. Baseline ML model: Linear Regression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Predict prices for unseen test data
predictions = model.predict(X_test)

# Evaluate the baseline model
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
baseline_mape = mean_absolute_percentage_error( y_test, predictions) * 100

print("BASELINE MODEL: LINEAR REGRESSION")
print("-" * 40)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", len(features))
print()
print(f"MAE:  ₹{mae:,.2f}")
print(f"RMSE: ₹{rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {baseline_mape:.2f}%")


BASELINE MODEL: LINEAR REGRESSION
----------------------------------------
Training rows: 670
Testing rows: 168
Features: 23

MAE:  ₹6,814.32
RMSE: ₹10,642.94
R²:   0.9601
MAPE: 19.32%


In [61]:

# 5. Dummy baseline: predict the training-set mean

dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

dummy_mae = mean_absolute_error(y_test, dummy_predictions)
dummy_r2 = r2_score(y_test, dummy_predictions)

print("DUMMY BASELINE")
print("-" * 40)
print(f"MAE: ₹{dummy_mae:,.2f}")
print(f"R²:  {dummy_r2:.4f}")


DUMMY BASELINE
----------------------------------------
MAE: ₹31,384.12
R²:  -0.0087


### Original-price vs. log-price target

I compared the original-price and log-price targets using five-fold cross-validation. The log-price model achieved lower MAE and MAPE, while the original-price model achieved higher R². I selected the log-price target because reducing relative prediction error was important for comparing phones across a wide range of prices.

In [62]:

# Evaluate models on the original price scale
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mape": "neg_mean_absolute_percentage_error"
}

print("Metrics configured.")
print("R²: higher is better")
print("MAE: lower is better")
print("MAPE: lower is better")


Metrics configured.
R²: higher is better
MAE: lower is better
MAPE: lower is better


In [63]:
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [64]:

# Original price target
plain_results = cross_validate(clone(model), X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Original-price model evaluated.")
print(f"Mean CV R²: {plain_results['test_r2'].mean():.4f}")
print(f"Mean CV MAE: ₹{-plain_results['test_mae'].mean():,.2f}")
print(f"Mean CV MAPE: {-plain_results['test_mape'].mean() * 100:.2f}%")


Original-price model evaluated.
Mean CV R²: 0.9042
Mean CV MAE: ₹7,554.62
Mean CV MAPE: 22.73%


In [65]:

# Log-transformed target
log_model = TransformedTargetRegressor(
    regressor=clone(model),
    func=np.log1p,
    inverse_func=np.expm1
)

log_results = cross_validate(log_model, X_train,y_train, cv=cv,scoring=scoring, n_jobs=-1)

print("Log-price model evaluated.")
print(f"Mean CV R²: {log_results['test_r2'].mean():.4f}")
print(f"Mean CV MAE: ₹{-log_results['test_mae'].mean():,.2f}")
print(f"Mean CV MAPE: {-log_results['test_mape'].mean() * 100:.2f}%")


Log-price model evaluated.
Mean CV R²: 0.8863
Mean CV MAE: ₹6,443.23
Mean CV MAPE: 13.56%


In [66]:

comparison = pd.DataFrame([
    {
        "Model": "Original price",
        "Mean R²": plain_results["test_r2"].mean(),
        "Mean MAE (INR)": -plain_results["test_mae"].mean(),
        "Mean MAPE (%)": -plain_results["test_mape"].mean() * 100
    },
    {
        "Model": "Log price",
        "Mean R²": log_results["test_r2"].mean(),
        "Mean MAE (INR)": -log_results["test_mae"].mean(),
        "Mean MAPE (%)": -log_results["test_mape"].mean() * 100
    }
])

print(comparison.round(4).to_string(index=False))

print("\nFold-by-fold R²:")
print(pd.DataFrame({
    "Fold": range(1, len(plain_results["test_r2"]) + 1),
    "Original price": plain_results["test_r2"],
    "Log price": log_results["test_r2"]
}).round(4).to_string(index=False))


         Model  Mean R²  Mean MAE (INR)  Mean MAPE (%)
Original price   0.9042       7554.6175        22.7273
     Log price   0.8863       6443.2327        13.5555

Fold-by-fold R²:
 Fold  Original price  Log price
    1          0.8633     0.8855
    2          0.9368     0.9189
    3          0.9457     0.9031
    4          0.9050     0.8530
    5          0.8701     0.8711


### Ridge
- alpha=1.0: a moderate starting regularization strength. It penalizes large coefficients and can reduce overfitting.
- We will tune alpha only if Ridge wins the initial comparison.
  
###  Random Forest
- n_estimators=300: enough trees for a reasonably stable starting estimate without excessive training time.
- min_samples_leaf=2: prevents leaves containing just one training observation, providing mild regularization.
- random_state=42: makes the experiment reproducible.
  
###  HistGradientBoosting
- max_iter=200: trains up to 200 boosting iterations.
- learning_rate=0.05: uses relatively small boosting steps to reduce aggressive fitting.
- max_leaf_nodes=15: limits each tree's complexity, useful for a dataset of roughly 838 in-scope phones.
  
###  XGBoost
- n_estimators=300: caps the number of boosting trees.
- learning_rate=0.05: makes each tree's contribution more conservative.
- max_depth=4: limits interaction complexity.
- subsample=0.8: uses 80% of training rows per boosting iteration.
- colsample_bytree=0.8: samples 80% of features per tree.
- reg_lambda=5: adds L2 regularization to discourage overly complex fits.
  XGBoost documentation explains that learning rate controls shrinkage and deeper trees increase complexity and overfitting risk. 
  
  
###  LightGBM
- n_estimators=300: caps the number of boosting iterations.
- learning_rate=0.05: conservative learning steps.
- num_leaves=15: restricts the number of terminal leaves per tree.
- max_depth=6: caps depth to help control complexity.
- min_child_samples=20: requires a reasonable number of observations in a leaf.
- reg_lambda=5: adds L2 regularization.

In [67]:
# Create a reusable pipeline with log-transformed target

def make_log_model(estimator):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("regressor", estimator)
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    )

In [68]:

candidate_models = {
    "Ridge": Ridge(
        alpha=1.0
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=6,
        min_child_samples=20,
        reg_lambda=5,
        verbosity=-1,
        random_state=42,
        n_jobs=1
    )
}


In [69]:

results = []

for name, estimator in candidate_models.items():
    candidate = make_log_model(clone(estimator))

    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Mean R2": scores["test_r2"].mean(),
        "Std R2": scores["test_r2"].std(ddof=1),
        "Mean MAE (INR)": -scores["test_mae"].mean(),
        "Mean MAPE (%)": -scores["test_mape"].mean() * 100
    })

comparison = (
    pd.DataFrame(results)
    .sort_values("Mean MAPE (%)")
    .reset_index(drop=True)
)

print(comparison.round(4).to_string(index=False))


                 Model  Mean R2  Std R2  Mean MAE (INR)  Mean MAPE (%)
                 Ridge   0.8902  0.0487       6260.8532        12.9853
               XGBoost   0.8822  0.0840       6486.0645        13.5141
              LightGBM   0.8672  0.0746       6844.8696        13.9850
Hist Gradient Boosting   0.8599  0.0741       7056.1647        14.2054
         Random Forest   0.8507  0.1099       7375.9626        14.7030


# Final Model Selection: Ridge vs. XGBoost

## Why compare these two models?

The initial five-fold cross-validation compared five regression models using the training data. Ridge achieved a mean R² of 0.8902 and mean MAPE of 12.9853%, while XGBoost achieved a mean R² of 0.8822 and mean MAPE of 13.5141%.

I selected Ridge and XGBoost for further tuning because they represent different modelling approaches:

- **Ridge Regression:** A regularized linear model that controls coefficient magnitude.
- **XGBoost:** A gradient-boosted tree model that can learn nonlinear relationships and interactions between smartphone specifications.

## Tuning strategy

Both models were tuned using GridSearchCV and five-fold cross-validation, with MAPE as the primary selection metric. Ridge Regression achieved a lower mean cross-validation MAPE than XGBoost, while XGBoost achieved a slightly higher mean R². Ridge was selected because it had lower relative prediction error and slightly lower MAE, with less variation in R² across folds. The selected model was then evaluated on the held-out test set.

In [70]:

# Final tuning: Ridge Regression

from sklearn.model_selection import GridSearchCV

ridge_pipeline = make_log_model(Ridge())

ridge_param_grid = {
    "regressor__regressor__alpha": np.logspace(-3, 3, 13)
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring=scoring,
    refit="mape",
    cv=cv,
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

print("Best Ridge parameters:", ridge_search.best_params_)

idx = ridge_search.best_index_

print(f"Best CV MAPE: {-ridge_search.best_score_ * 100:.2f}%")
print(f"CV R²: {ridge_search.cv_results_['mean_test_r2'][idx]:.4f}")
print(
    f"CV MAE: ₹"
    f"{-ridge_search.cv_results_['mean_test_mae'][idx]:,.2f}"
)


Best Ridge parameters: {'regressor__regressor__alpha': np.float64(0.31622776601683794)}
Best CV MAPE: 12.98%
CV R²: 0.8878
CV MAE: ₹6,259.90


In [71]:

# Final tuning: XGBoost

xgb_pipeline = make_log_model(
    clone(candidate_models["XGBoost"])
)

xgb_param_grid = {
    "regressor__regressor__n_estimators": [200, 400],
    "regressor__regressor__learning_rate": [0.03, 0.05],
    "regressor__regressor__max_depth": [2, 3, 4],
    "regressor__regressor__subsample": [0.8, 1.0],
    "regressor__regressor__colsample_bytree": [0.8, 1.0],
    "regressor__regressor__reg_lambda": [1, 5]
}

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring=scoring,
    refit="mape",
    cv=cv,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

print("Best XGBoost parameters:", xgb_search.best_params_)

idx = xgb_search.best_index_

print(f"Best CV MAPE: {-xgb_search.best_score_ * 100:.2f}%")
print(f"CV R²: {xgb_search.cv_results_['mean_test_r2'][idx]:.4f}")
print(
    f"CV MAE: ₹"
    f"{-xgb_search.cv_results_['mean_test_mae'][idx]:,.2f}"
)


Best XGBoost parameters: {'regressor__regressor__colsample_bytree': 0.8, 'regressor__regressor__learning_rate': 0.05, 'regressor__regressor__max_depth': 4, 'regressor__regressor__n_estimators': 400, 'regressor__regressor__reg_lambda': 1, 'regressor__regressor__subsample': 0.8}
Best CV MAPE: 13.10%
CV R²: 0.8892
CV MAE: ₹6,289.78


In [72]:

# Final cross-validation comparison after tuning

searches = {
    "Ridge (tuned)": ridge_search,
    "XGBoost (tuned)": xgb_search
}

tuned_results = []

for name, search in searches.items():
    idx = search.best_index_
    cv_results = search.cv_results_

    tuned_results.append({
        "Model": name,
        "Mean R2": cv_results["mean_test_r2"][idx],
        "Std R2": cv_results["std_test_r2"][idx],
        "Mean MAE (INR)": -cv_results["mean_test_mae"][idx],
        "Mean MAPE (%)": -cv_results["mean_test_mape"][idx] * 100
    })

tuned_comparison = (
    pd.DataFrame(tuned_results)
    .sort_values("Mean MAPE (%)")
    .reset_index(drop=True)
)

print(tuned_comparison.round(4).to_string(index=False))


          Model  Mean R2  Std R2  Mean MAE (INR)  Mean MAPE (%)
  Ridge (tuned)   0.8878  0.0466       6259.9017        12.9794
XGBoost (tuned)   0.8892  0.0718       6289.7774        13.0981


In [73]:
# Evaluate the selected model on the held-out test set

if ridge_search.best_score_ >= xgb_search.best_score_:
    final_model = ridge_search.best_estimator_
    selected_search = ridge_search
    selected_model_name = "Ridge Regression"
else:
    final_model = xgb_search.best_estimator_
    selected_search = xgb_search
    selected_model_name = "XGBoost"

# Evaluate the selected model on the untouched test set
test_predictions = final_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)
test_mape = mean_absolute_percentage_error(
    y_test, test_predictions
) * 100

print(f"FINAL MODEL: {selected_model_name}")
print("-" * 40)
print("Best parameters:", selected_search.best_params_)
print(f"Test R²:   {test_r2:.4f}")
print(f"Test MAE:  ₹{test_mae:,.2f}")
print(f"Test RMSE: ₹{test_rmse:,.2f}")
print(f"Test MAPE: {test_mape:.2f}%")


FINAL MODEL: Ridge Regression
----------------------------------------
Best parameters: {'regressor__regressor__alpha': np.float64(0.31622776601683794)}
Test R²:   0.9508
Test MAE:  ₹6,437.00
Test RMSE: ₹11,817.66
Test MAPE: 13.24%


In [74]:
# Calculate prediction errors
# Error analysis: individual test predictions

error_df = pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "predicted_price": test_predictions
})

# Residual = actual - predicted
error_df["residual"] = (
    error_df["actual_price"] - error_df["predicted_price"]
)

error_df["absolute_error"] = error_df["residual"].abs()

error_df["absolute_percentage_error"] = (
    error_df["absolute_error"] / error_df["actual_price"] * 100
)

print("Test prediction error summary:")
print(
    error_df[
        ["actual_price", "predicted_price", "residual",
         "absolute_error", "absolute_percentage_error"]
    ].head(10).round(2).to_string(index=False)
)

print("\nLargest prediction errors:")
print(
    error_df.nlargest(10, "absolute_error")[
        ["actual_price", "predicted_price", "residual", "absolute_error"]
    ].round(2).to_string(index=False)
)




Test prediction error summary:
 actual_price  predicted_price  residual  absolute_error  absolute_percentage_error
        32149         30504.42   1644.58         1644.58                       5.12
        31999         27950.27   4048.73         4048.73                      12.65
        31470         33326.14  -1856.14         1856.14                       5.90
        14999         16300.17  -1301.17         1301.17                       8.68
        12249         14416.32  -2167.32         2167.32                      17.69
        28990         20333.75   8656.25         8656.25                      29.86
        17999         32459.16 -14460.16        14460.16                      80.34
        22499         21725.12    773.88          773.88                       3.44
        32499         40969.78  -8470.78         8470.78                      26.06
        64999         64522.37    476.63          476.63                       0.73

Largest prediction errors:
 actual_price  pr

In [75]:
# Evaluate performance by price segment

# Error analysis by actual price segment

error_df["price_segment"] = pd.cut(
    error_df["actual_price"],
    bins=[0, 10000, 25000, 50000, float("inf")],
    labels=["Budget (≤₹10k)", "Mid (₹10k–25k)",
            "Upper-mid (₹25k–50k)", "Premium (>₹50k)"],
    include_lowest=True
)

segment_results = (
    error_df.groupby("price_segment", observed=False)
    .agg(
        phones=("actual_price", "count"),
        median_actual_price=("actual_price", "median"),
        MAE=("absolute_error", "mean"),
        MAPE=("absolute_percentage_error", "mean")
    )
    .reset_index()
)

print(segment_results.round(2).to_string(index=False))


       price_segment  phones  median_actual_price      MAE  MAPE
      Budget (≤₹10k)      10               9444.5  1847.17 20.63
      Mid (₹10k–25k)      53              17442.0  2283.45 12.53
Upper-mid (₹25k–50k)      64              34494.5  4402.99 12.03
     Premium (>₹50k)      41              89999.0 16100.75 14.25


# Final Model Selection and Error Analysis

## Model Selection

Five regression models were compared using five-fold cross-validation on the training data. Ridge Regression and XGBoost were selected for hyperparameter tuning using GridSearchCV.

Ridge Regression was selected as the final model based on mean cross-validation MAPE. The tuned Ridge model achieved a mean CV MAPE of 12.98% and a mean CV MAE of ₹6,259.90. XGBoost achieved a mean CV MAPE of 13.10%. Although XGBoost obtained a slightly higher mean CV R², Ridge performed better on the primary selection metric and had lower fold-to-fold R² variability.

## Final Test Performance

The selected Ridge Regression model was evaluated on the held-out test set.

| Metric | Test Result |
|---|---:|
| R² | 0.9508 |
| MAE | ₹6,437.00 |
| RMSE | ₹11,817.66 |
| MAPE | 13.24% |

The model achieved an R² of 0.9508, with an average absolute prediction error of approximately ₹6,437 on the test set. The MAPE of 13.24% provides a relative measure of prediction error across the range of phone prices.

## Error Analysis by Price Segment

| Price Segment | Test Phones | MAE | MAPE |
|---|---:|---:|---:|
| Budget (≤₹10,000) | 10 | ₹1,847.17 | 20.63% |
| Mid-range (₹10,000–₹25,000) | 53 | ₹2,283.45 | 12.53% |
| Upper-mid-range (₹25,000–₹50,000) | 64 | ₹4,402.99 | 12.03% |
| Premium (>₹50,000) | 41 | ₹16,100.75 | 14.25% |

Relative prediction error was highest in the budget segment, although that segment contained only 10 test phones. The premium segment had the highest absolute error, consistent with the higher prices in that segment. These results show why model performance should be examined across price bands rather than through a single overall metric.

## Limitations

- The model predicts Smartprix listed prices from the period when the data was collected. These may differ from current retail prices.
- The test-set results are based on a random split of the collected records; a grouped evaluation by phone family or series would provide a further test of generalization to unseen models.
- The model uses Smartprix's `spec_score`, which may not be available when predicting prices from other data sources or in a real deployment setting.

